# Очистка датасета

In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import dask.dataframe as dd

In [2]:
df = pd.read_csv("/home/danila/metrolog/backup/expands.csv")

In [3]:
df.head()

,miinstance_passport,miinstance_name,miinstance_type,miinstance_state_condition,miinstance_tech_condition,issue_date,commissioning_date,is_fit,MPI
0,14120,Манометр избыточного давления показывающий,МП3-УУ2,В эксплуатации,Годен,01.06.2001 0:00:00,NaN,True,12.0
1,14121,Манометр избыточного давления показывающий,МП3-УУ2,В эксплуатации,Годен,01.06.2001 0:00:00,NaN,False,12.0
2,14122,Манометр избыточного давления показывающий,МП3-УУ2,В эксплуатации,Годен,01.06.2001 0:00:00,NaN,True,12.0
3,14123,Манометр избыточного давления показывающий,МП3-УУ2,В обменном фонде,Годен,01.06.2001 0:00:00,NaN,False,12.0
4,14124,Манометр избыточного давления показывающий,МП3-УУ2,В эксплуатации,Годен,01.06.2001 0:00:00,NaN,True,12.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265336 entries, 0 to 265335
Data columns (total 9 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   miinstance_passport         265336 non-null  int64  
 1   miinstance_name             265336 non-null  object 
 2   miinstance_type             265336 non-null  object 
 3   miinstance_state_condition  263246 non-null  object 
 4   miinstance_tech_condition   262751 non-null  object 
 5   issue_date                  256476 non-null  object 
 6   commissioning_date          27128 non-null   object 
 7   is_fit                      233823 non-null  object 
 8   MPI                         246686 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 18.2+ MB


In [5]:
print(df.isnull().sum())

miinstance_passport                0
miinstance_name                    0
miinstance_type                    0
miinstance_state_condition      2090
miinstance_tech_condition       2585
issue_date                      8860
commissioning_date            238208
is_fit                         31513
MPI                            18650
dtype: int64


In [6]:
## Выводим кол-во значений типа null для очистки
print(df.isnull().sum() / len(df) * 100)

miinstance_passport            0.000000
miinstance_name                0.000000
miinstance_type                0.000000
miinstance_state_condition     0.787681
miinstance_tech_condition      0.974236
issue_date                     3.339162
commissioning_date            89.775982
is_fit                        11.876639
MPI                            7.028824
dtype: float64


In [7]:
## Чистим датасет от null
## Т.к. у столбца commissioning_date почти 90% null, столбец смысла не имеет, удаляем
df = df.drop("commissioning_date", axis = 1)

ddf = dd.from_pandas(df, npartitions=4)
means = ddf.mean(numeric_only=True).compute()
df = df.fillna(means)

## строковые значения заполняем модой
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    mode_val = df[col].mode()
    if len(mode_val) > 0:  # проверяем, что мода существует
        df[col] = df[col].fillna(mode_val[0])

## выыводим кол-во null
print(df.isnull().sum() / len(df) * 100)

miinstance_passport           0.0
miinstance_name               0.0
miinstance_type               0.0
miinstance_state_condition    0.0
miinstance_tech_condition     0.0
issue_date                    0.0
is_fit                        0.0
MPI                           0.0
dtype: float64


/tmp/ipykernel_26098/3668451315.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_val[0])
